https://adk.dev/get-started/python/

Google ADK requiere que tengamos una variable de entorno que se llame "MODELO_API_KEY", donde "MODELO" es el modelo que vamos a usar (GEMINI para Gemini, ANTHROPIC para Anthropic, etc.). Automáticamente nos detecta la API Key y no es necesaria pasársela al modelo. Esto es bueno ya que nos evita problemas de API Key Leakages

In [28]:
import os
from dotenv import load_dotenv
load_dotenv()

os.environ["GEMINI_API_KEY"] = os.getenv("GEMINI_API_KEY1")

# Crear el agente

Crear un agente con Google ADK es tan sencillo como sigue

In [ ]:
from google.adk.agents import LlmAgent

# Primero definimos algunas herramientas para que el agente pueda usarlas
def sumar(a: int, b: int) -> int:
    "Herramienta para sumar dos números"
    return a + b

def restar(a:int, b: int) -> int:
    "Herramienta para restar dos números"
    return a - b

root_agent = LlmAgent(
    name = "MB3_AGENT", # Parámetro obligatorio y útil cuando se tengan muchos agentes
    model = "gemini-3.1-flash-lite-preview",
    description = "Agente de pruebas", # Parámetro útil cuando se tengan muchos agentes
    instruction = "Responde con buenas maneras",
    tools = [sumar, restar]
)

Para usar el agente, hay que crear un Runner. Además de esto, hay que establecer la forma en la que queremos que ADK nos gestione la memoria. Se define una sesión como "a series of interactions between a user and agents". Esta sesión puede vivir en nuestra RAM (lo cual no es persistente) o en una Base de Datos o nube (lo cual garantiza que la sesión se mantiene incluso si retomamos la conversación otro día, por ejemplo)

# InMemorySessionService

Si establecemos este método, la memoria del agente solo vive lo que dura la aplicación; si, por ejemplo, retomamos la conversación mañana, el agente no tendrá memoria de la conversación. Esta forma solo es útil para pruebas, no para despliegues

In [15]:
from google.adk.runners import Runner
from google.adk.sessions import InMemorySessionService
from google.genai import types

app_name = "Agente_matemático" # Identificador de la aplicación
session_service = InMemorySessionService() # Guardamos en la RAM el historial de conversación
user_id = "user123" # Identificador del usuario
runner = Runner(
    agent = root_agent,
    app_name = app_name,
    session_service = session_service
)

# Listamos todas las sessiones que el usuario tiene en la aplicación 
existing_sessions = await session_service.list_sessions(
    app_name = app_name,
    user_id = user_id
)
print("Sessiones existentes -> ",existing_sessions)

# Si es la primera vez que el usuario usa la aplicación, hay que crear la sessión para que pueda interactuar
if existing_sessions and len(existing_sessions.sessions) > 0:
    session_id = existing_sessions.sessions[-1].id
    print(f"Usamos la sessión más reciente: {session_id}")
else:
    new_session = await session_service.create_session(
        app_name = app_name,
        user_id = user_id
    )
    session_id = new_session.id
    print(f"Creamos una nueva sessión para el usuario")

# Listamos todas las sessiones que el usuario tiene en la aplicación 
existing_sessions = await session_service.list_sessions(
    app_name = app_name,
    user_id = user_id
)
print("Sessiones existentes -> ", existing_sessions)    

Sessiones existentes ->  sessions=[]
Creamos una nueva sessión para el usuario
Sessiones existentes ->  sessions=[Session(id='0dded791-76ef-4c04-a46f-348d1925abf9', app_name='Agente_matemático', user_id='user123', state={}, events=[], last_update_time=1776067782.894399)]


Una vez creada una sessión, estamos listos para establecer una conversación

In [33]:
while True:
    user_input = input("Tú: ")
    if user_input == "exit":
        break
    content = types.Content(role="user", parts=[types.Part(text=user_input)])

    async for event in runner.run_async(
        user_id = user_id,
        session_id = session_id,
        new_message = content
    ):
        print("USER_INPUT", user_input)
        print("EVENTOS\n", event)
        print("-"*50)
        print("-"*50)

USER_INPUT Hola, me llamo Arthas. Cuánto es 23423 + 2342423 - 432543?
EVENTOS
 model_version='gemini-3.1-flash-lite-preview' content=Content(
  parts=[
    Part(
      function_call=FunctionCall(
        args={
          'a': 23423,
          'b': 2342423
        },
        id='axksgrfm',
        name='sumar'
      ),
      thought_signature=b'\x124\n2\x01\xbe>\xf6\xfb\xba\x81\xb1>p\x8d*Tt%\xc9\xb2)\x12\xda\x88\x85\xfa\x10\xaaa\xdf\xac(1\xfa\xbd;\xc1\xc5\xe1Y\x9a\xe0\xe7\x02\xc1\xbd\x94vs\x17\xa7\xfe\xa2'
    ),
  ],
  role='model'
) grounding_metadata=None partial=None turn_complete=None finish_reason=<FinishReason.STOP: 'STOP'> error_code=None error_message=None interrupted=None custom_metadata=None usage_metadata=GenerateContentResponseUsageMetadata(
  candidates_token_count=27,
  prompt_token_count=426,
  prompt_tokens_details=[
    ModalityTokenCount(
      modality=<MediaModality.TEXT: 'TEXT'>,
      token_count=426
    ),
  ],
  total_token_count=453
) live_session_resumption_up